# RoboTwin × LingBot-VLA-v2 on ROCm

This notebook is the interactive entry point for the `external-data` image. Run cells selectively: model serving and evaluation are long-running GPU workloads. The platform-provided data must be available at `/models/robotwin-persistent`. Runtime outputs are written to `/workspace/runtime`.

In [1]:
from pathlib import Path
import os, signal, subprocess, time, urllib.request

ROBOTWIN = Path('/RoboTwin')
RUNTIME = Path('/workspace/runtime')
PYTHON = '/opt/robotwin-env/bin/python'
MODEL = ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/models/robbyant_lingbot-vla-v2-6b-robotwin/checkpoints/global_step_50000/hf_ckpt'
RUNTIME.joinpath('outputs/logs').mkdir(parents=True, exist_ok=True)
print('RoboTwin:', ROBOTWIN)
print('Model:', MODEL)

RoboTwin: /RoboTwin
Model: /RoboTwin/experiments/lingbot_vla_v2_6b_robotwin/models/robbyant_lingbot-vla-v2-6b-robotwin/checkpoints/global_step_50000/hf_ckpt


## 1. Environment and mounted-data check

In [2]:
required = [
    ROBOTWIN / 'assets/objects/objaverse/list.json',
    ROBOTWIN / 'data/demo_clean',
    ROBOTWIN / 'data/lerobot',
    MODEL / 'model.safetensors.index.json',
]
for path in required:
    assert path.exists(), f'Missing: {path}'
subprocess.run([PYTHON, '-c', "import torch; print('torch=', torch.__version__, 'hip=', torch.version.hip, 'gpu=', torch.cuda.is_available(), 'count=', torch.cuda.device_count())"], check=True)

torch= 2.9.1+gitff65f5b hip= 7.2.53211-e1a6bc5663 gpu= True count= 4


CompletedProcess(args=['/opt/robotwin-env/bin/python', '-c', "import torch; print('torch=', torch.__version__, 'hip=', torch.version.hip, 'gpu=', torch.cuda.is_available(), 'count=', torch.cuda.device_count())"], returncode=0)

### Expected svulkan2 warning on AMD

During SAPIEN initialization, the following messages are expected on ROCm/AMD and can be ignored if Vulkan rendering and evaluation continue:

```text
[svulkan2] [error] CUDA Error: cudaErrorInsufficientDriver
[svulkan2] [error] Failed to initialize denoiser
```

svulkan2 is probing its optional NVIDIA CUDA denoiser; RoboTwin continues through the Vulkan renderer. Investigate `/dev/dri`, `PYOPENGL_PLATFORM=egl`, and `vulkaninfo --summary` only if the process subsequently exits or camera images are missing.

## 2. Start the model server on port 13400

Stop any existing process using port 13400 before running this cell. The server log is written under `/workspace/runtime/outputs/logs`.

In [3]:
server_log = RUNTIME / 'outputs/logs/official_server.log'
server_command = [
    'bash', str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/launch_official_server.sh'),
    '0', '13400', str(server_log), 'False', str(MODEL),
]
server_log_handle = server_log.open('ab')
SERVER_PROCESS = subprocess.Popen(server_command, cwd=ROBOTWIN, stdout=server_log_handle, stderr=subprocess.STDOUT, start_new_session=True)
print('server pid:', SERVER_PROCESS.pid, 'log:', server_log)
for _ in range(600):
    if SERVER_PROCESS.poll() is not None:
        raise RuntimeError(f'Server exited with code {SERVER_PROCESS.returncode}; inspect {server_log}')
    try:
        urllib.request.urlopen('http://127.0.0.1:13400/healthz', timeout=2).read()
        print('server ready: 127.0.0.1:13400')
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError(f'Server did not become ready; inspect {server_log}')

server pid: 6971 log: /workspace/runtime/outputs/logs/official_server.log


server ready: 127.0.0.1:13400


## 3. Run 10 closed-loop `adjust_bottle` episodes

> **Important: the following two lines do not indicate an evaluation failure and can be ignored.** svulkan2 is probing an optional CUDA denoiser that is available only on NVIDIA systems. These messages are expected on AMD ROCm, and RoboTwin will continue with Vulkan rendering:
>
> ```text
> [2026-08-26 12:03:21.547] [svulkan2] [error] CUDA Error: cudaErrorInsufficientDriver
> [2026-08-26 12:03:21.547] [svulkan2] [error] Failed to initialize denoiser
> ```
> Investigate the Vulkan and device configuration only if the evaluation process subsequently exits or camera images are missing.

In [4]:
EVAL_EPISODES = 10
eval_command = [
    PYTHON, str(ROBOTWIN / 'scripts/eval_policy_xpolicylab.py'),
    '--task_name', 'adjust_bottle', '--task_config', 'demo_clean',
    '--policy_name', 'LingBot-VLA-v2', '--protocol', 'lingbot_vla_v2',
    '--host', '127.0.0.1', '--port', '13400', '--device_id', '0',
    '--seed', '0', '--test_num', str(EVAL_EPISODES), '--expert_check', 'false',
    '--eval_batch', 'false',
]
eval_env = os.environ.copy()
eval_env.update(ROBOTWIN_DISABLE_CUROBO='1', ROBOTWIN_EE_PLANNER='mplib', PYOPENGL_PLATFORM='egl')
base_eval_started = time.perf_counter()
subprocess.run(eval_command, cwd=ROBOTWIN, env=eval_env, check=True)
base_eval_elapsed = time.perf_counter() - base_eval_started
print(f'base evaluation total: {base_eval_elapsed:.2f}s ({base_eval_elapsed / 60:.2f} min)')
print(f'base evaluation average per episode: {base_eval_elapsed / EVAL_EPISODES:.2f}s')

Traceback (most recent call last):
  File "/RoboTwin/envs/robot/planner.py", line 15, in <module>
    from curobo.types.math import Pose as CuroboPose
ModuleNotFoundError: No module named 'curobo'


[2026-08-31 05:22:43.852] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:22:43.852] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:22:43.852] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:22:43.852] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:22:43.853] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:22:43.853] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:22:43.853] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:22:43.853] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:22:43.853] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:22:43.853] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:22:43.853] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:22:43.853] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:22:43.853] [svulkan

[2026-08-31 05:22:44.378] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:22:44.378] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:22:44.378] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:22:44.378] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:22:44.378] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:22:44.378] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:22:44.378] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:22:44.378] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:22:44.378] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:22:44.378] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:22:44.378] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:22:44.378] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:22:44.379] [svulkan

[2026-08-31 05:23:06.642] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:23:06.642] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:23:06.642] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:23:06.642] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:23:06.642] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:23:06.642] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:23:06.642] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:23:06.642] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:23:06.642] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:23:06.642] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:23:06.642] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:23:06.642] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:23:06.642] [svulkan

[2026-08-31 05:23:18.910] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:23:18.910] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:23:18.911] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:23:18.911] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:23:18.911] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:23:18.911] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:23:18.911] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:23:18.911] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:23:18.911] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:23:18.911] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:23:18.911] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:23:18.911] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:23:18.911] [svulkan

[2026-08-31 05:23:19.435] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:23:19.435] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:23:19.435] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:23:19.435] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:23:19.435] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:23:19.435] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:23:19.436] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:23:19.436] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:23:19.436] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:23:19.436] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:23:19.436] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:23:19.436] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:23:19.436] [svulkan

... (14 outputs left out)

[2026-08-31 05:25:13.446] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:25:13.446] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:25:13.446] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:25:13.446] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:25:13.446] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:25:13.446] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:25:13.446] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:25:13.446] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:25:13.447] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:25:13.447] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:25:13.447] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:25:13.447] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:25:13.447] [svulkan

[benchmark_timing] seed=100012 calls=5 mean_s=0.8097 p50_s=0.8091 p95_s=0.8138 total_s=4.0485
Success!
adjust_bottle | LingBot-VLA-v2 | demo_clean | default
Success rate: 7/8 => 87.5%, current seed: 100012

Episode 0: No valid instructions found
Failed to generate episode instruction; using task name as instruction.
[benchmark_timing] seed=100013 calls=5 mean_s=0.8096 p50_s=0.8072 p95_s=0.8195 total_s=4.0479
Success!
adjust_bottle | LingBot-VLA-v2 | demo_clean | default
Success rate: 8/9 => 88.9%, current seed: 100013

Episode 0: No valid instructions found
Failed to generate episode instruction; using task name as instruction.


[benchmark_timing] seed=100015 calls=5 mean_s=0.8097 p50_s=0.8094 p95_s=0.8111 total_s=4.0484
Success!
adjust_bottle | LingBot-VLA-v2 | demo_clean | default
Success rate: 9/10 => 90.0%, current seed: 100015

Data has been saved to eval_result/adjust_bottle/LingBot-VLA-v2/demo_clean/default/2026-08-31_05-22-43_316594/_result.txt
Final success rate: 9/10 = 90.0%


base evaluation total: 168.87s (2.81 min)
base evaluation average per episode: 16.89s


## 4. LoRA training, checkpoint merge, and inference

Choose `GPU_COUNT` as 1, 2, or 4 and set `TRAIN_STEPS` explicitly. The default 100 steps is a smoke test for the complete pipeline, not an expectation of useful fine-tuning quality. Start with 1,000–5,000 steps for an experiment and select the final value from held-out closed-loop evaluation. The effective global batch size remains 4. This cell stops the model server started above before training so it releases GPU memory.

In [5]:
if 'SERVER_PROCESS' in globals() and SERVER_PROCESS.poll() is None:
    os.killpg(os.getpgid(SERVER_PROCESS.pid), signal.SIGTERM)
    SERVER_PROCESS.wait(timeout=30)
    server_log_handle.close()
    print('model server stopped before training')

GPU_COUNT = 1  # Supported values: 1, 2, 4
TRAIN_STEPS = 100  # Smoke test; increase to e.g. 1000 or 5000 for fine-tuning
if GPU_COUNT not in (1, 2, 4):
    raise ValueError('GPU_COUNT must be 1, 2, or 4')
if TRAIN_STEPS <= 0:
    raise ValueError('TRAIN_STEPS must be positive')
gradient_accumulation_steps = 4 // GPU_COUNT
training_root = ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin'
source_root = training_root / 'source/lingbot-vla-v2'
training_yaml = training_root / 'training/reproduction_100steps/lingbotvla_cli.yaml'
training_output = RUNTIME / f'outputs/reproduction_{TRAIN_STEPS}steps'
train_command = [
    PYTHON, '-m', 'torch.distributed.run', '--standalone',
    f'--nproc-per-node={GPU_COUNT}', '-m', 'tasks.vla.train_lingbotvla',
    str(training_yaml),
    '--train.data_parallel_shard_size', str(GPU_COUNT),
    '--train.gradient_accumulation_steps', str(gradient_accumulation_steps),
    '--train.max_steps', str(TRAIN_STEPS),
    '--train.save_steps', str(TRAIN_STEPS),
    '--train.output_dir', str(training_output),
]
train_env = os.environ.copy()
train_env['HIP_VISIBLE_DEVICES'] = ','.join(str(i) for i in range(GPU_COUNT))
train_env.pop('ROCR_VISIBLE_DEVICES', None)
train_env.pop('CUDA_VISIBLE_DEVICES', None)
training_started = time.perf_counter()
subprocess.run(train_command, cwd=source_root, env=train_env, check=True)
training_elapsed = time.perf_counter() - training_started
print(f'LoRA training total: {training_elapsed:.2f}s ({training_elapsed / 60:.2f} min, {training_elapsed / 3600:.2f} h)')
print(f'LoRA wall-clock average per optimizer step: {training_elapsed / TRAIN_STEPS:.2f}s')

model server stopped before training


/opt/robotwin-env/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/opt/robotwin-env/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias usi

[aiter] WARNING: NUMA balancing is enabled, which may cause errors. It is recommended to disable NUMA balancing by running "sudo sh -c 'echo 0 > /proc/sys/kernel/numa_balancing'" for more details: https://rocm.docs.amd.com/en/latest/how-to/system-optimization/mi300x.html#disable-numa-auto-balancing
[aiter] import [module_aiter_core] under /opt/aiter/aiter/jit/module_aiter_core.so


/opt/aiter/aiter/ops/deepgemm.py:22: RuntimeWarning: aiter.ops.opus (a16w16) is gfx950-only; detected arch='gfx1100'. opus_gemm_* calls will raise RuntimeError at invocation. opus_gemm uses gfx950-only intrinsics (MFMA, ds_read_b64_tr) and the 160 KiB LDS budget. Set GPU_ARCHS=gfx950 (or run on a gfx950 device) to use this module.
  from .opus.gemm_op_a16w16 import opus_gemm_a16w16_tune as _opus_tune
/opt/aiter/aiter/ops/triton/gluon/__init__.py:16: UserWarning: [aiter] AITER_USE_SYSTEM_TRITON=1: aiter gluon kernels require triton>=3.6.0, found 3.5.1. Please install a compatible version via .github/scripts/install_triton.sh, otherwise unexpected errors may occur.
  warnings.warn(


Overwriting feature type 'VideoFrame' (VideoFrame -> VideoFrame)


Load MoGe Module Failed!!
[INFO][lingbotvla.utils.arguments:114] 08/31/2026 05:25:38 >> gradient_accumulation_steps=4, global_batch_size=4
08/31/2026 05:25:38 - INFO - __main__ - Process rank: 0, world size: 1
08/31/2026 05:25:38 - INFO - __main__ - {
  "model": {
    "config_path": "/RoboTwin/experiments/lingbot_vla_v2_6b_robotwin/models/robbyant_lingbot-vla-v2-6b-robotwin/checkpoints/global_step_50000/hf_ckpt",
    "model_path": "/RoboTwin/experiments/lingbot_vla_v2_6b_robotwin/models/robbyant_lingbot-vla-v2-6b-robotwin/checkpoints/global_step_50000/hf_ckpt",
    "tokenizer_path": "/RoboTwin/experiments/lingbot_vla_v2_6b_robotwin/models/Qwen3-VL-4B-Instruct-config-tokenizer",
    "vlm_repo_id": null,
    "post_training": false,
    "vocab_size": 0,
    "incremental_training": false,
    "adanorm_time": true,
    "encoders": {},
    "decoders": {},
    "input_encoder": "encoder",
    "output_encoder": "decoder",
    "encode_target": false,
    "attn_implementation": "flash_attention_2

... (235 outputs left out)

[INFO][lingbotvla.checkpoint.checkpointer:114] 08/31/2026 05:38:54 >> Saved checkpoint to /workspace/runtime/outputs/reproduction_100steps/checkpoints/global_step_100
08/31/2026 05:38:54 - INFO - __main__ - Distributed checkpoint saved at /workspace/runtime/outputs/reproduction_100steps/checkpoints/global_step_100 successfully!
08/31/2026 05:38:54 - INFO - __main__ - Reached max_steps=100, stopping training.
[INFO][lingbotvla.utils.helper:114] 08/31/2026 05:38:54 >> VRAM usage after epoch 1: cur 23.59GB, max 24.78GB.


/opt/venv/lib/python3.12/site-packages/torch/distributed/distributed_c10d.py:4876: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  warnings.warn(  # warn only once
Step: 100/100 [12:57<00:00,  7.77s/it]


08/31/2026 05:38:54 - INFO - __main__ - [async_hf] summary: {"total": 0, "hf_success": 0, "hf_failed": 0, "eval_submitted": 0, "eval_skipped": 0, "eval_failed": 0, "failures": []}


LoRA training total: 807.93s (13.47 min, 0.22 h)
LoRA wall-clock average per optimizer step: 8.08s


### 4.1 Merge the LoRA checkpoint

In [6]:
checkpoint = training_output / f'checkpoints/global_step_{TRAIN_STEPS}'
MERGED = training_output / f'merged_checkpoint/global_step_{TRAIN_STEPS}/hf_ckpt'
assert checkpoint.exists(), f'Missing checkpoint: {checkpoint}'
merge_command = [
    PYTHON, str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/merge_lora_dcp.py'),
    '--checkpoint', str(checkpoint),
    '--training-output', str(training_output),
    '--base-model', str(MODEL), '--output', str(MERGED),
    '--rank', '8', '--alpha', '16',
]
subprocess.run(merge_command, cwd=ROBOTWIN, check=True)
assert (MERGED / 'model.safetensors.index.json').exists(), f'Merge output is incomplete: {MERGED}'
print('merged model:', MERGED)

[aiter] WARNING: NUMA balancing is enabled, which may cause errors. It is recommended to disable NUMA balancing by running "sudo sh -c 'echo 0 > /proc/sys/kernel/numa_balancing'" for more details: https://rocm.docs.amd.com/en/latest/how-to/system-optimization/mi300x.html#disable-numa-auto-balancing
[aiter] import [module_aiter_core] under /opt/aiter/aiter/jit/module_aiter_core.so


/opt/aiter/aiter/ops/deepgemm.py:22: RuntimeWarning: aiter.ops.opus (a16w16) is gfx950-only; detected arch='gfx1100'. opus_gemm_* calls will raise RuntimeError at invocation. opus_gemm uses gfx950-only intrinsics (MFMA, ds_read_b64_tr) and the 160 KiB LDS budget. Set GPU_ARCHS=gfx950 (or run on a gfx950 device) to use this module.
  from .opus.gemm_op_a16w16 import opus_gemm_a16w16_tune as _opus_tune


/opt/aiter/aiter/ops/triton/gluon/__init__.py:16: UserWarning: [aiter] AITER_USE_SYSTEM_TRITON=1: aiter gluon kernels require triton>=3.6.0, found 3.5.1. Please install a compatible version via .github/scripts/install_triton.sh, otherwise unexpected errors may occur.
  warnings.warn(


[INFO][lingbotvla.models.module_utils:450] 08/31/2026 05:39:41 >> Model weight splits saved in /workspace/runtime/outputs/reproduction_100steps/merged_checkpoint/global_step_100/hf_ckpt.


Merged 144 LoRA layers into /workspace/runtime/outputs/reproduction_100steps/merged_checkpoint/global_step_100/hf_ckpt


merged model: /workspace/runtime/outputs/reproduction_100steps/merged_checkpoint/global_step_100/hf_ckpt


### 4.2 Start the merged LoRA model on port 13400

The original server was stopped before training, so the merged model reuses the same endpoint.

In [7]:
merged_log = RUNTIME / 'outputs/logs/merged_server.log'
merged_server_command = [
    'bash', str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/launch_official_server.sh'),
    '0', '13400', str(merged_log), 'False', str(MERGED),
]
merged_log_handle = merged_log.open('ab')
MERGED_SERVER_PROCESS = subprocess.Popen(merged_server_command, cwd=ROBOTWIN, stdout=merged_log_handle, stderr=subprocess.STDOUT, start_new_session=True)
print('merged server pid:', MERGED_SERVER_PROCESS.pid, 'log:', merged_log)
for _ in range(600):
    if MERGED_SERVER_PROCESS.poll() is not None:
        raise RuntimeError(f'Merged server exited with code {MERGED_SERVER_PROCESS.returncode}; inspect {merged_log}')
    try:
        urllib.request.urlopen('http://127.0.0.1:13400/healthz', timeout=2).read()
        print('merged server ready: 127.0.0.1:13400')
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError(f'Merged server did not become ready; inspect {merged_log}')

merged server pid: 105006 log: /workspace/runtime/outputs/logs/merged_server.log


merged server ready: 127.0.0.1:13400


### 4.3 Validate the merged LoRA model with 10 closed-loop episodes

In [8]:
merged_eval_started = time.perf_counter()
subprocess.run(eval_command, cwd=ROBOTWIN, env=eval_env, check=True)
merged_eval_elapsed = time.perf_counter() - merged_eval_started
print(f'merged evaluation total: {merged_eval_elapsed:.2f}s ({merged_eval_elapsed / 60:.2f} min)')
print(f'merged evaluation average per episode: {merged_eval_elapsed / EVAL_EPISODES:.2f}s')

Traceback (most recent call last):
  File "/RoboTwin/envs/robot/planner.py", line 15, in <module>
    from curobo.types.math import Pose as CuroboPose
ModuleNotFoundError: No module named 'curobo'


[2026-08-31 05:40:41.806] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:40:41.806] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:40:41.806] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:40:41.806] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:40:41.807] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:40:41.807] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:40:41.807] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:40:41.807] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:40:41.807] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:40:41.807] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:40:41.807] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:40:41.807] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:40:41.807] [svulkan

[2026-08-31 05:40:42.334] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:40:42.334] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:40:42.334] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:40:42.334] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:40:42.334] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:40:42.334] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:40:42.334] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:40:42.334] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:40:42.334] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:40:42.334] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:40:42.334] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:40:42.334] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:40:42.334] [svulkan

Render Well
============= Config =============

Messy Table: False
Random Background: False
Random Light: False
Random Table Height: 0
Random Head Camera Distance: 0
Head Camera Config: D435, True
Wrist Camera Config: D435, True
Embodiment Config: aloha-agilex
Eval Instruction: seen

[planner.py]: Something wrong happened when importing CuroboPlanner! Please check if Curobo is installed correctly. If the problem still exists, you can install Curobo from https://github.com/NVlabs/curobo manually.
Exception traceback:
missing pytorch3d
Task Name: adjust_bottle
Policy Name: LingBot-VLA-v2
Episode 0: No valid instructions found
Failed to generate episode instruction; using task name as instruction.


[2026-08-31 05:41:26.112] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:41:26.112] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:41:26.112] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:41:26.112] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:41:26.113] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:41:26.113] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:41:26.113] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:41:26.113] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:41:26.113] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:41:26.113] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:41:26.113] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:41:26.113] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:41:26.113] [svulkan

[benchmark_timing] seed=100001 calls=16 mean_s=1.2829 p50_s=0.8774 p95_s=3.0035 total_s=20.5261
Fail!
adjust_bottle | LingBot-VLA-v2 | demo_clean | default
Success rate: 0/1 => 0.0%, current seed: 100001

Episode 0: No valid instructions found
Failed to generate episode instruction; using task name as instruction.


... (18 outputs left out)

[2026-08-31 05:46:01.430] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:46:01.430] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:46:01.430] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:46:01.430] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:46:01.430] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:46:01.430] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:46:01.431] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:46:01.431] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:46:01.431] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:46:01.431] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:46:01.431] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:46:01.431] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:46:01.431] [svulkan

[benchmark_timing] seed=100013 calls=16 mean_s=1.3910 p50_s=0.8015 p95_s=3.6092 total_s=22.2560
Fail!
adjust_bottle | LingBot-VLA-v2 | demo_clean | default
Success rate: 2/9 => 22.2%, current seed: 100013

Episode 0: No valid instructions found
Failed to generate episode instruction; using task name as instruction.


[benchmark_timing] seed=100015 calls=16 mean_s=1.1833 p50_s=0.8303 p95_s=2.5646 total_s=18.9326
Fail!
adjust_bottle | LingBot-VLA-v2 | demo_clean | default
Success rate: 2/10 => 20.0%, current seed: 100015

Data has been saved to eval_result/adjust_bottle/LingBot-VLA-v2/demo_clean/default/2026-08-31_05-40-41_263060/_result.txt
Final success rate: 2/10 = 20.0%


merged evaluation total: 365.97s (6.10 min)
merged evaluation average per episode: 36.60s


### 4.4 Optional: stop the merged LoRA model server

In [9]:
if 'MERGED_SERVER_PROCESS' in globals() and MERGED_SERVER_PROCESS.poll() is None:
    os.killpg(os.getpgid(MERGED_SERVER_PROCESS.pid), signal.SIGTERM)
    MERGED_SERVER_PROCESS.wait(timeout=30)
    merged_log_handle.close()
    print('merged model server stopped')
else:
    print('no merged server started by this notebook kernel')

merged model server stopped


## 5. Full-parameter SFT, checkpoint conversion, and inference

This is independent from the LoRA workflow above: it sets `use_lora=false` and updates all trainable model parameters. `FULL_SFT_GPU_COUNT` supports 1, 2, or 4 and defaults to 4. The script keeps global batch 4 by using gradient accumulation 4, 2, or 1 respectively. Start with `FULL_SFT_STEPS = 1`; each full DCP checkpoint directory is roughly 70 GB in total across all ranks, not 70 GB per GPU.

Under FSDP2, `FULL_SFT_ENABLE_FULL_SHARD=True` means `reshard_after_forward=True`: each wrapped module is sharded again after forward, lowering peak VRAM at the cost of another all-gather before backward. `False` keeps that module's full parameters through backward, which may be faster but generally uses more peak VRAM. It does not disable FSDP or replicate the entire model on every GPU. Keep `True` unless a one-step comparison proves that `False` fits safely.

The available four-GPU configurations are: (1) the default full-shard run below with shard/global batch 4; (2) the same run with no reshard for the communication-versus-memory comparison; and (3) a full-shard run with `enable_fp32` and `use_future_image` enabled. They are alternatives, not consecutive training stages.

In [10]:
# Start with 1 step. GPU count may be 1, 2, or 4; the default is 4.
FULL_SFT_GPU_COUNT = 4
FULL_SFT_STEPS = 1
FULL_SFT_SAVE_STEPS = FULL_SFT_STEPS
FULL_SFT_ENABLE_FULL_SHARD = True  # Recommended lower-memory default
FULL_SFT_OUTPUT = RUNTIME / f'outputs/full_sft_{FULL_SFT_GPU_COUNT}gpu_{FULL_SFT_STEPS}steps'

for process_name in ('SERVER_PROCESS', 'MERGED_SERVER_PROCESS'):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        os.killpg(os.getpgid(process.pid), signal.SIGTERM)
        process.wait(timeout=30)

gpu_count = int(subprocess.check_output(
    [PYTHON, '-c', 'import torch; print(torch.cuda.device_count())'], text=True
).strip())
assert FULL_SFT_GPU_COUNT in (1, 2, 4), 'FULL_SFT_GPU_COUNT must be 1, 2, or 4'
assert gpu_count >= FULL_SFT_GPU_COUNT, f'Full SFT requested {FULL_SFT_GPU_COUNT} GPUs, found {gpu_count}'

full_sft_env = os.environ.copy()
full_sft_env.update({
    'GPU_COUNT': str(FULL_SFT_GPU_COUNT),
    'MAX_STEPS': str(FULL_SFT_STEPS),
    'SAVE_STEPS': str(FULL_SFT_SAVE_STEPS),
    'ENABLE_FULL_SHARD': str(FULL_SFT_ENABLE_FULL_SHARD).lower(),
    'OUTPUT_DIR': str(FULL_SFT_OUTPUT),
})
full_sft_command = [
    'bash', str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/training/train_full_sft.sh')
]
full_sft_started = time.perf_counter()
subprocess.run(full_sft_command, cwd=ROBOTWIN, env=full_sft_env, check=True)
full_sft_elapsed = time.perf_counter() - full_sft_started
print(f'full SFT total: {full_sft_elapsed:.2f}s ({full_sft_elapsed / 60:.2f} min)')
print('checkpoint:', FULL_SFT_OUTPUT / f'checkpoints/global_step_{FULL_SFT_STEPS}')


*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************


/opt/robotwin-env/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/opt/robotwin-env/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias usi

[aiter] WARNING: NUMA balancing is enabled, which may cause errors. It is recommended to disable NUMA balancing by running "sudo sh -c 'echo 0 > /proc/sys/kernel/numa_balancing'" for more details: https://rocm.docs.amd.com/en/latest/how-to/system-optimization/mi300x.html#disable-numa-auto-balancing
[aiter] import [module_aiter_core] under /opt/aiter/aiter/jit/module_aiter_core.so
[aiter] WARNING: NUMA balancing is enabled, which may cause errors. It is recommended to disable NUMA balancing by running "sudo sh -c 'echo 0 > /proc/sys/kernel/numa_balancing'" for more details: https://rocm.docs.amd.com/en/latest/how-to/system-optimization/mi300x.html#disable-numa-auto-balancing
[aiter] import [module_aiter_core] under /opt/aiter/aiter/jit/module_aiter_core.so
[aiter] WARNING: NUMA balancing is enabled, which may cause errors. It is recommended to disable NUMA balancing by running "sudo sh -c 'echo 0 > /proc/sys/kernel/numa_balancing'" for more details: https://rocm.docs.amd.com/en/latest/h

/opt/aiter/aiter/ops/deepgemm.py:22: RuntimeWarning: aiter.ops.opus (a16w16) is gfx950-only; detected arch='gfx1100'. opus_gemm_* calls will raise RuntimeError at invocation. opus_gemm uses gfx950-only intrinsics (MFMA, ds_read_b64_tr) and the 160 KiB LDS budget. Set GPU_ARCHS=gfx950 (or run on a gfx950 device) to use this module.
  from .opus.gemm_op_a16w16 import opus_gemm_a16w16_tune as _opus_tune
/opt/aiter/aiter/ops/deepgemm.py:22: RuntimeWarning: aiter.ops.opus (a16w16) is gfx950-only; detected arch='gfx1100'. opus_gemm_* calls will raise RuntimeError at invocation. opus_gemm uses gfx950-only intrinsics (MFMA, ds_read_b64_tr) and the 160 KiB LDS budget. Set GPU_ARCHS=gfx950 (or run on a gfx950 device) to use this module.
  from .opus.gemm_op_a16w16 import opus_gemm_a16w16_tune as _opus_tune
/opt/aiter/aiter/ops/triton/gluon/__init__.py:16: UserWarning: [aiter] AITER_USE_SYSTEM_TRITON=1: aiter gluon kernels require triton>=3.6.0, found 3.5.1. Please install a compatible version vi

/opt/aiter/aiter/ops/deepgemm.py:22: RuntimeWarning: aiter.ops.opus (a16w16) is gfx950-only; detected arch='gfx1100'. opus_gemm_* calls will raise RuntimeError at invocation. opus_gemm uses gfx950-only intrinsics (MFMA, ds_read_b64_tr) and the 160 KiB LDS budget. Set GPU_ARCHS=gfx950 (or run on a gfx950 device) to use this module.
  from .opus.gemm_op_a16w16 import opus_gemm_a16w16_tune as _opus_tune
/opt/aiter/aiter/ops/deepgemm.py:22: RuntimeWarning: aiter.ops.opus (a16w16) is gfx950-only; detected arch='gfx1100'. opus_gemm_* calls will raise RuntimeError at invocation. opus_gemm uses gfx950-only intrinsics (MFMA, ds_read_b64_tr) and the 160 KiB LDS budget. Set GPU_ARCHS=gfx950 (or run on a gfx950 device) to use this module.
  from .opus.gemm_op_a16w16 import opus_gemm_a16w16_tune as _opus_tune
/opt/aiter/aiter/ops/triton/gluon/__init__.py:16: UserWarning: [aiter] AITER_USE_SYSTEM_TRITON=1: aiter gluon kernels require triton>=3.6.0, found 3.5.1. Please install a compatible version vi

Overwriting feature type 'VideoFrame' (VideoFrame -> VideoFrame)
Overwriting feature type 'VideoFrame' (VideoFrame -> VideoFrame)
Load MoGe Module Failed!!
08/31/2026 05:46:54 - INFO - __main__ - Process rank: 3, world size: 4


... (39 outputs left out)

/opt/venv/lib/python3.12/site-packages/torch/distributed/checkpoint/filesystem.py:133: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  if tensor.storage().size() != tensor.numel():
/opt/venv/lib/python3.12/site-packages/torch/distributed/checkpoint/filesystem.py:133: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  if tensor.storage().size() != tensor.numel():
/opt/venv/lib/python3.12/site-packages/torch/distributed/checkpoint/filesystem.py:133: UserWarning: TypedStorage is deprecated. It will be removed in the future

[INFO][lingbotvla.checkpoint.checkpointer:114] 08/31/2026 05:47:50 >> Saved checkpoint to /workspace/runtime/outputs/full_sft_4gpu_1steps/checkpoints/global_step_1
/opt/venv/lib/python3.12/site-packages/torch/distributed/distributed_c10d.py:4876: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  warnings.warn(  # warn only once
08/31/2026 05:47:51 - INFO - __main__ - Distributed checkpoint saved at /workspace/runtime/outputs/full_sft_4gpu_1steps/checkpoints/global_step_1 successfully!
08/31/2026 05:47:51 - INFO - __main__ - Reached max_steps=1, stopping training.
[INFO][lingbotvla.utils.helper:114] 08/31/2026 05:47:51 >> VRAM usage after epoch 1: cur 17.50GB, max 25.24GB.
Step: 1/1 [00:36<00:00, 36.81s/it]


08/31/2026 05:47:51 - INFO - __main__ - [async_hf] summary: {"total": 0, "hf_success": 0, "hf_failed": 0, "eval_submitted": 0, "eval_skipped": 0, "eval_failed": 0, "failures": []}


full SFT total: 69.41s (1.16 min)
checkpoint: /workspace/runtime/outputs/full_sft_4gpu_1steps/checkpoints/global_step_1


### 5.1 Convert the full-SFT DCP checkpoint

Full SFT has no LoRA adapter to merge. This step gathers the distributed DCP model state and writes a directly loadable Hugging Face checkpoint. Do not use `merge_lora_dcp.py` here.

In [11]:
FULL_SFT_CHECKPOINT = FULL_SFT_OUTPUT / f'checkpoints/global_step_{FULL_SFT_STEPS}'
FULL_SFT_MODEL = FULL_SFT_OUTPUT / f'merged_checkpoint/global_step_{FULL_SFT_STEPS}/hf_ckpt'
assert FULL_SFT_CHECKPOINT.exists(), f'Missing checkpoint: {FULL_SFT_CHECKPOINT}'
full_sft_convert_command = [
    PYTHON, str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/convert_full_sft_dcp.py'),
    '--checkpoint', str(FULL_SFT_CHECKPOINT),
    '--training-output', str(FULL_SFT_OUTPUT),
    '--output', str(FULL_SFT_MODEL),
]
convert_started = time.perf_counter()
subprocess.run(full_sft_convert_command, cwd=ROBOTWIN, check=True)
convert_elapsed = time.perf_counter() - convert_started
assert (FULL_SFT_MODEL / 'model.safetensors.index.json').exists(), f'Conversion is incomplete: {FULL_SFT_MODEL}'
print(f'full-SFT conversion total: {convert_elapsed:.2f}s ({convert_elapsed / 60:.2f} min)')
print('full-SFT model:', FULL_SFT_MODEL)

[aiter] WARNING: NUMA balancing is enabled, which may cause errors. It is recommended to disable NUMA balancing by running "sudo sh -c 'echo 0 > /proc/sys/kernel/numa_balancing'" for more details: https://rocm.docs.amd.com/en/latest/how-to/system-optimization/mi300x.html#disable-numa-auto-balancing
[aiter] import [module_aiter_core] under /opt/aiter/aiter/jit/module_aiter_core.so


/opt/aiter/aiter/ops/deepgemm.py:22: RuntimeWarning: aiter.ops.opus (a16w16) is gfx950-only; detected arch='gfx1100'. opus_gemm_* calls will raise RuntimeError at invocation. opus_gemm uses gfx950-only intrinsics (MFMA, ds_read_b64_tr) and the 160 KiB LDS budget. Set GPU_ARCHS=gfx950 (or run on a gfx950 device) to use this module.
  from .opus.gemm_op_a16w16 import opus_gemm_a16w16_tune as _opus_tune


/opt/aiter/aiter/ops/triton/gluon/__init__.py:16: UserWarning: [aiter] AITER_USE_SYSTEM_TRITON=1: aiter gluon kernels require triton>=3.6.0, found 3.5.1. Please install a compatible version via .github/scripts/install_triton.sh, otherwise unexpected errors may occur.
  warnings.warn(


[INFO][lingbotvla.models.module_utils:450] 08/31/2026 05:48:38 >> Model weight splits saved in /workspace/runtime/outputs/full_sft_4gpu_1steps/merged_checkpoint/global_step_1/hf_ckpt.


Converted full-SFT checkpoint to /workspace/runtime/outputs/full_sft_4gpu_1steps/merged_checkpoint/global_step_1/hf_ckpt


full-SFT conversion total: 45.54s (0.76 min)
full-SFT model: /workspace/runtime/outputs/full_sft_4gpu_1steps/merged_checkpoint/global_step_1/hf_ckpt


### 5.2 Start and validate the converted full-SFT model on port 13400

In [12]:
full_sft_server_log = RUNTIME / 'outputs/logs/full_sft_server.log'
full_sft_server_command = [
    'bash', str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/launch_official_server.sh'),
    '0', '13400', str(full_sft_server_log), 'False', str(FULL_SFT_MODEL),
]
full_sft_server_log_handle = full_sft_server_log.open('ab')
FULL_SFT_SERVER_PROCESS = subprocess.Popen(full_sft_server_command, cwd=ROBOTWIN, stdout=full_sft_server_log_handle, stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(600):
    if FULL_SFT_SERVER_PROCESS.poll() is not None:
        raise RuntimeError(f'Full-SFT server exited with code {FULL_SFT_SERVER_PROCESS.returncode}; inspect {full_sft_server_log}')
    try:
        urllib.request.urlopen('http://127.0.0.1:13400/healthz', timeout=2).read()
        print('full-SFT server ready: 127.0.0.1:13400')
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError(f'Full-SFT server did not become ready; inspect {full_sft_server_log}')

full_sft_eval_started = time.perf_counter()
subprocess.run(eval_command, cwd=ROBOTWIN, env=eval_env, check=True)
full_sft_eval_elapsed = time.perf_counter() - full_sft_eval_started
print(f'full-SFT evaluation total: {full_sft_eval_elapsed:.2f}s ({full_sft_eval_elapsed / 60:.2f} min)')
print(f'full-SFT evaluation average per episode: {full_sft_eval_elapsed / EVAL_EPISODES:.2f}s')

full-SFT server ready: 127.0.0.1:13400


Traceback (most recent call last):
  File "/RoboTwin/envs/robot/planner.py", line 15, in <module>
    from curobo.types.math import Pose as CuroboPose
ModuleNotFoundError: No module named 'curobo'


[2026-08-31 05:49:38.511] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:49:38.511] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:49:38.511] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:49:38.511] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:49:38.511] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:49:38.511] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:49:38.511] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:49:38.511] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:49:38.511] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:49:38.511] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:49:38.511] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:49:38.511] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:49:38.512] [svulkan

[2026-08-31 05:49:39.039] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:49:39.039] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:49:39.039] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:49:39.039] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:49:39.039] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:49:39.039] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:49:39.039] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:49:39.039] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:49:39.039] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:49:39.039] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:49:39.039] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:49:39.039] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:49:39.040] [svulkan

Render Well
============= Config =============

Messy Table: False
Random Background: False
Random Light: False
Random Table Height: 0
Random Head Camera Distance: 0
Head Camera Config: D435, True
Wrist Camera Config: D435, True
Embodiment Config: aloha-agilex
Eval Instruction: seen

[planner.py]: Something wrong happened when importing CuroboPlanner! Please check if Curobo is installed correctly. If the problem still exists, you can install Curobo from https://github.com/NVlabs/curobo manually.
Exception traceback:
missing pytorch3d
Task Name: adjust_bottle
Policy Name: LingBot-VLA-v2
Episode 0: No valid instructions found
Failed to generate episode instruction; using task name as instruction.


[2026-08-31 05:50:20.854] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:50:20.854] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:50:20.854] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:50:20.854] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:50:20.854] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:50:20.854] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:50:20.854] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:50:20.854] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:50:20.854] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:50:20.854] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:50:20.854] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:50:20.854] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:50:20.854] [svulkan

... (20 outputs left out)

[2026-08-31 05:55:57.934] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:55:57.934] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:55:57.934] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:55:57.934] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:55:57.934] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:55:57.934] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:55:57.934] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:55:57.934] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:55:57.935] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:55:57.935] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:55:57.935] [svulkan2] [error]  CUDA Error: cudaErrorInsufficientDriver
[2026-08-31 05:55:57.935] [svulkan2] [error] Failed to initialize denoiser
[2026-08-31 05:55:57.935] [svulkan

[benchmark_timing] seed=100013 calls=16 mean_s=1.0828 p50_s=0.8797 p95_s=1.7027 total_s=17.3241
Fail!
adjust_bottle | LingBot-VLA-v2 | demo_clean | default
Success rate: 0/9 => 0.0%, current seed: 100013

Episode 0: No valid instructions found
Failed to generate episode instruction; using task name as instruction.


[benchmark_timing] seed=100015 calls=16 mean_s=0.9889 p50_s=0.8127 p95_s=1.7643 total_s=15.8217
Fail!
adjust_bottle | LingBot-VLA-v2 | demo_clean | default
Success rate: 0/10 => 0.0%, current seed: 100015

Data has been saved to eval_result/adjust_bottle/LingBot-VLA-v2/demo_clean/default/2026-08-31_05-49-37_889509/_result.txt
Final success rate: 0/10 = 0.0%


full-SFT evaluation total: 425.32s (7.09 min)
full-SFT evaluation average per episode: 42.53s


### 5.3 Optional: stop the full-SFT model server

In [13]:
if 'FULL_SFT_SERVER_PROCESS' in globals() and FULL_SFT_SERVER_PROCESS.poll() is None:
    os.killpg(os.getpgid(FULL_SFT_SERVER_PROCESS.pid), signal.SIGTERM)
    FULL_SFT_SERVER_PROCESS.wait(timeout=30)
    full_sft_server_log_handle.close()
    print('full-SFT model server stopped')
else:
    print('no full-SFT server started by this notebook kernel')

full-SFT model server stopped
